# YOLO26n Cane V1 — Training and Export

## Goal

Train a deterministic YOLO26n smoke run or Cane V1 dataset, validate the best checkpoint, export ONNX and NCNN artifacts, and compare validation metrics. Colab values prove workflow correctness and model quality only; they are not Raspberry Pi performance measurements.

## Setup

For the checked-in smoke validation, keep `DATASET = 'coco8.yaml'` and `EPOCHS = 1`. For V1, upload the harmonized dataset archive, extract it under `/content/cane-v1`, set `DATASET` to its `data.yaml`, and choose the reviewed training duration.

In [ ]:
# @title 1. Install the pinned major-version runtime
%pip install -q 'ultralytics==8.4.132'

In [ ]:
# @title 2. Parameters
MODEL = 'yolo26n.pt'
DATASET = 'coco8.yaml'
EPOCHS = 1
IMAGE_SIZE = 320
BATCH = 8
SEED = 42
RUN_NAME = 'notebook-gpu-smoke'

In [ ]:
# @title 3. Verify hardware and runtime
import json, platform, shutil
from pathlib import Path
import torch, ultralytics
from ultralytics import YOLO

assert torch.cuda.is_available(), 'This notebook validation requires a GPU runtime'
DEVICE = 0
print({'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__, 'ultralytics': ultralytics.__version__})

## Steps

In [ ]:
# @title 4. Train deterministically
PROJECT = Path('/content/yolo-pi-notebook-runs')
model = YOLO(MODEL)
train_result = model.train(data=DATASET, epochs=EPOCHS, imgsz=IMAGE_SIZE, batch=BATCH, device=DEVICE, workers=2, project=str(PROJECT), name=RUN_NAME, exist_ok=True, seed=SEED, deterministic=True, plots=True, verbose=False)
BEST = Path(train_result.save_dir) / 'weights' / 'best.pt'
assert BEST.exists(), BEST
BEST

In [ ]:
# @title 5. Validate and export
best_model = YOLO(str(BEST))
pt_metrics = best_model.val(data=DATASET, imgsz=IMAGE_SIZE, device=DEVICE, split='val', plots=False, verbose=False)
ONNX = Path(YOLO(str(BEST)).export(format='onnx', imgsz=IMAGE_SIZE, batch=1, device='cpu'))
NCNN = Path(YOLO(str(BEST)).export(format='ncnn', imgsz=IMAGE_SIZE, batch=1, device='cpu', end2end=False, quantize=16))
assert ONNX.exists() and NCNN.exists()
ONNX, NCNN

## Checks

In [ ]:
# @title 6. Check exported-model metric parity
def metrics_dict(metrics):
    return {str(k): float(v) for k, v in metrics.results_dict.items() if isinstance(v, (int, float))}

onnx_metrics = YOLO(str(ONNX)).val(data=DATASET, imgsz=IMAGE_SIZE, device='cpu', split='val', plots=False, verbose=False)
ncnn_metrics = YOLO(str(NCNN)).val(data=DATASET, imgsz=IMAGE_SIZE, device='cpu', split='val', plots=False, verbose=False)
SUMMARY = {'purpose': 'workflow validation; not Pi performance', 'gpu': torch.cuda.get_device_name(0), 'model': MODEL, 'dataset': DATASET, 'epochs': EPOCHS, 'imgsz': IMAGE_SIZE, 'seed': SEED, 'metrics': {'pytorch': metrics_dict(pt_metrics), 'onnx': metrics_dict(onnx_metrics), 'ncnn': metrics_dict(ncnn_metrics)}}
SUMMARY_PATH = PROJECT / RUN_NAME / 'validation-summary.json'
SUMMARY_PATH.write_text(json.dumps(SUMMARY, indent=2, sort_keys=True) + '\n')
SUMMARY

In [ ]:
# @title 7. Package artifacts
ARCHIVE = Path(shutil.make_archive('/content/yolo-pi-notebook-artifacts', 'zip', PROJECT))
print({'archive': str(ARCHIVE), 'summary': str(SUMMARY_PATH)})

## Next Steps

1. Download the archive before stopping the Colab session.
2. For V1, review the harmonizer report and manually inspect every class.
3. Record model-quality metrics separately from Pi runtime metrics.
4. Validate all exported artifacts again on the Raspberry Pi using the fixed benchmark manifest.